# 🗑️ Waste Image Classification — MobileNetV2 + Grad-CAM

**Kelompok DL Semester 4 · 2026**

### Urutan eksekusi
| Cell | Fungsi | Catatan |
|------|--------|---------|
| 1 | SETUP — fix deps | Wajib restart runtime setelah ini |
| 2 | CLONE — ambil repo | Idempotent |
| 3 | VERIFY — cek imports | Idempotent |
| 4 | DATASET — download data | Skip jika sudah ada |
| 5 | PREFLIGHT — validasi env | GPU = warning, bukan stop |
| 6 | TRAIN — jalankan training | CPU auto-reduce batch |
| 7 | VERIFY — cek model output | Idempotent |
| 8 | EVAL — evaluasi model | Idempotent |
| 9 | GRADCAM — visualisasi | Output ke outputs/ |
| 10 | API — test inference | Optional |
| 11 | DOWNLOAD — zip + download | Final step |

In [1]:
# ============================================================
# CELL 1: SETUP — Fix ml_dtypes + install dependencies
# Run this FIRST. Restart runtime after this cell finishes.
# ============================================================
import subprocess, sys

def _pip(*args):
    return subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *args],
        capture_output=True, text=True
    )

print("🔧 Step 1/3 — Upgrading ml_dtypes BEFORE any imports...")
r = _pip("ml_dtypes>=0.5", "--upgrade")
print("  ✅ Done" if r.returncode == 0 else f"  ⚠️  {r.stderr[-200:]}")

print("\n📦 Step 2/3 — Installing project requirements...")
import os
req_file = "requirements-dev.txt" if os.path.exists("requirements-dev.txt") else "requirements.txt"
r2 = _pip("-r", req_file)
if r2.returncode == 0:
    print(f"  ✅ {req_file} installed")
else:
    print(f"  ⚠️  Some packages failed:\n{r2.stderr[-500:]}")

print("\n🔧 Step 3/3 — Re-upgrading ml_dtypes (in case it was downgraded)...")
r3 = _pip("ml_dtypes>=0.5", "--upgrade")
print("  ✅ Done" if r3.returncode == 0 else f"  ⚠️  {r3.stderr[-200:]}")

print()
print("=" * 65)
print("  ⚠️  PENTING: Runtime → Restart session, lalu lanjut ke Cell 2")
print("=" * 65)

🔧 Step 1/3 — Upgrading ml_dtypes BEFORE any imports...
  ✅ Done

📦 Step 2/3 — Installing project requirements...
  ⚠️  Some packages failed:
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


🔧 Step 3/3 — Re-upgrading ml_dtypes (in case it was downgraded)...
  ✅ Done

  ⚠️  PENTING: Runtime → Restart session, lalu lanjut ke Cell 2


In [1]:
# ============================================================
# CELL 2: CLONE — Clone / pull repo dari GitHub
# Idempotent: safe untuk di-rerun
# ============================================================
import os, subprocess, sys

REPO_URL = "https://github.com/ne-he/Image-Classification-DL.git"
REPO_DIR = "/content/Image-Classification-DL"

if os.path.exists(os.path.join(REPO_DIR, ".git")):
    print("📁 Repo sudah ada — pulling latest...")
    r = subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"],
                       capture_output=True, text=True)
    print(r.stdout.strip() or "Already up to date.")
    if r.returncode != 0:
        print(f"  ⚠️  Pull warning: {r.stderr.strip()}")
else:
    print(f"📥 Cloning {REPO_URL}...")
    r = subprocess.run(["git", "clone", REPO_URL, REPO_DIR],
                       capture_output=True, text=True)
    if r.returncode == 0:
        print("  ✅ Clone berhasil")
    else:
        print(f"  ❌ Clone gagal: {r.stderr}")
        raise RuntimeError("Git clone failed — cek REPO_URL atau koneksi internet")

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"\n📂 Working directory: {os.getcwd()}")
print("✅ Repo siap — lanjut ke Cell 3")

📁 Repo sudah ada — pulling latest...
Already up to date.

📂 Working directory: /content/Image-Classification-DL
✅ Repo siap — lanjut ke Cell 3


In [2]:
# ============================================================
# CELL 3: VERIFY IMPORTS
# Pakai importlib bukan __import__ untuk hindari RecursionError
# Idempotent: safe untuk di-rerun
# ============================================================
import sys, importlib

sys.setrecursionlimit(10000)

LIBS = [
    "tensorflow",
    "numpy",
    "cv2",
    "sklearn",
    "PIL",
    "fastapi",
    "yaml",
    "tqdm",
    "matplotlib",
    "seaborn",
]

print("🔍 Verifying imports...\n")
failed = []

for lib in LIBS:
    # Purge stale cache first
    for key in list(sys.modules.keys()):
        if key == lib or key.startswith(lib + "."):
            del sys.modules[key]
    try:
        mod = importlib.import_module(lib)
        ver = getattr(mod, "__version__", "ok")
        print(f"  ✅ {lib:<20} {ver}")
    except RecursionError:
        print(f"  ⚠️  {lib:<20} RecursionError — coba Runtime → Restart")
        failed.append(lib)
    except Exception as exc:
        print(f"  ❌ {lib:<20} {exc}")
        failed.append(lib)

print()
if not failed:
    print("✅ Semua imports OK — lanjut ke Cell 4")
else:
    print(f"⚠️  {len(failed)} library gagal: {failed}")
    print("   Biasanya cukup dengan: Runtime → Restart session lalu rerun Cell 2–3")

🔍 Verifying imports...

  ✅ tensorflow           2.20.0
  ✅ numpy                2.0.2
  ❌ cv2                  module 'cv2.dnn' has no attribute 'DictValue'
  ✅ sklearn              1.6.1
  ✅ PIL                  11.3.0


/usr/lib/python3.12/importlib/__init__.py:90: UserWarning: The NumPy module was reloaded (imported a second time). This can in some cases result in small but subtle issues and is discouraged.
  return _bootstrap._gcd_import(name[level:], package, level)


  ✅ fastapi              0.136.1
  ✅ yaml                 6.0.3
  ✅ tqdm                 4.67.3
  ✅ matplotlib           3.10.0
  ✅ seaborn              0.13.2

⚠️  1 library gagal: ['cv2']
   Biasanya cukup dengan: Runtime → Restart session lalu rerun Cell 2–3


In [ ]:
# ============================================================
# CELL 4: DATASET — Download dataset (HuggingFace + fallback)
# Idempotent: skip jika dataset sudah ada (>100 images)
# ============================================================
import os, sys, subprocess, importlib
from pathlib import Path

DATASET_DIR = Path("./data/dataset-resized")
DATASET_DIR.mkdir(parents=True, exist_ok=True)

# --- Check if dataset exists ---
existing_classes = [d for d in DATASET_DIR.iterdir() if d.is_dir()]
total_images = sum(len(list(d.glob("*.*"))) for d in existing_classes)

print(f"📊 Dataset check: {len(existing_classes)} classes, {total_images} images")

if total_images > 100:
    print(f"✅ Dataset sudah ada — skip download")
else:
    print("📥 Attempting HuggingFace download (garythung/trashnet)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "datasets", "-q"],
                   capture_output=True)

    try:
        datasets_lib = importlib.import_module("datasets")
        PIL_Image = importlib.import_module("PIL.Image")

        dataset = datasets_lib.load_dataset("garythung/trashnet", trust_remote_code=True)
        label_names = dataset["train"].features["label"].names
        print(f"  ✅ Dataset loaded: {len(label_names)} classes — {label_names}")

        for split_name, split_data in dataset.items():
            for i, sample in enumerate(split_data):
                label = label_names[sample["label"]]
                out_dir = DATASET_DIR / label
                out_dir.mkdir(parents=True, exist_ok=True)
                img = sample["image"].resize((224, 224))
                img.save(out_dir / f"{split_name}_{i:05d}.jpg")
            count = len(split_data)
            print(f"  ✅ {split_name}: {count} images saved")

    except Exception as exc:
        print(f"  ⚠️  HuggingFace download gagal: {exc}")
        print()
        print("  📋 Upload manual via Google Drive:")
        print("     from google.colab import drive")
        print("     drive.mount('/content/drive')")
        print("     !cp -r '/content/drive/MyDrive/dataset-resized' ./data/")
        print()
        print("  Atau via Kaggle — set KAGGLE_USERNAME + KAGGLE_KEY lalu:")
        print("     !kaggle datasets download -d <dataset-slug> -p ./data/")

# --- Final summary ---
classes = [d for d in DATASET_DIR.iterdir() if d.is_dir()]
total = sum(len(list(d.glob("*.*"))) for d in classes)
print(f"\n📊 Dataset final: {len(classes)} classes, {total} images")
for c in sorted(classes):
    n = len(list(c.glob("*.*")))
    print(f"  {c.name:<25} {n:>5} images")

📊 Dataset check: 0 classes, 0 images
📥 Attempting HuggingFace download (garythung/trashnet)...


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'garythung/trashnet' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'garythung/trashnet' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to re

README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

dataset-original.zip:   0%|          | 0.00/3.63G [00:00<?, ?B/s]

dataset-resized.zip:   0%|          | 0.00/42.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5054 [00:00<?, ? examples/s]

  ✅ Dataset loaded: 6 classes — ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']


In [ ]:
# ============================================================
# CELL 5: PREFLIGHT — Environment validation
# GPU = WARNING only, bukan blocking stop
# ============================================================
import importlib
from pathlib import Path

print("=" * 60)
print("🔍 PRE-FLIGHT CHECK")
print("=" * 60)

checks = []  # (name, ok: True/False/None, message)

# --- 1. GPU (WARNING only) ---
tf = importlib.import_module("tensorflow")
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    checks.append(("GPU", True, f"{len(gpus)} device(s) — FAST training mode"))
else:
    checks.append(("GPU", None,
                   "Not found — CPU mode (batch=16, ~6 min/epoch)"))
    print("  💡 Tip: Runtime → Change runtime type → T4 GPU (10× faster)")

# --- 2. Dataset ---
dataset_path = Path("./data/dataset-resized")
classes = [d for d in dataset_path.iterdir() if d.is_dir()] if dataset_path.exists() else []
total_imgs = sum(len(list(d.glob("*.*"))) for d in classes)
if total_imgs > 100:
    checks.append(("Dataset", True, f"{len(classes)} classes, {total_imgs} images"))
else:
    checks.append(("Dataset", False, f"Missing at {dataset_path} — run Cell 4 first"))

# --- 3. Config YAML ---
cfg_path = Path("./configs/train_mobilenet.yaml")
checks.append(("Config YAML", cfg_path.exists(), str(cfg_path)))

# --- 4. Source code ---
src_path = Path("./src")
checks.append(("Source code", src_path.exists(), str(src_path)))

# --- 5. Train script ---
train_script = Path("./scripts/train.py")
checks.append(("Train script", train_script.exists(), str(train_script)))

# --- 6. Existing checkpoint (info only) ---
model_dir = Path("./models/final")
ckpts = sorted(model_dir.glob("mobilenet_v2_*/best_model.keras")) if model_dir.exists() else []
if ckpts:
    latest = ckpts[-1]
    checks.append(("Checkpoint", None, f"Found: {latest.parent.name}"))
else:
    checks.append(("Checkpoint", None, "None — fresh training"))

# --- Print results ---
print()
blocking_fail = False
for name, status, msg in checks:
    if status is True:
        icon = "✅"
    elif status is None:
        icon = "⚠️ "
    else:
        icon = "❌"
        blocking_fail = True
    print(f"  {icon} {name:<22} {msg}")

print()
if blocking_fail:
    print("❌ STOP — ada check yang gagal (❌). Fix dulu sebelum lanjut.")
else:
    print("✅ Pre-flight OK — lanjut ke Cell 6 untuk training")

In [ ]:
# ============================================================
# CELL 6: TRAIN — Run training (CPU/GPU auto-detect)
# train.py sudah handle GPU detection + reduce batch for CPU
# ============================================================
import subprocess, sys
from pathlib import Path

model_dir = Path("./models/final")

# Show existing checkpoints
ckpts = sorted(model_dir.glob("mobilenet_v2_*/best_model.keras")) if model_dir.exists() else []
if ckpts:
    print(f"💾 Existing checkpoint(s) found:")
    for c in ckpts:
        print(f"   {c.parent.name}")
    print("   Training will create a NEW run alongside these.")

print("\n🚀 Starting training...")
print("   (Output streamed live — this may take a while on CPU)")
print("-" * 60)

result = subprocess.run(
    [sys.executable, "scripts/train.py", "--config", "configs/train_mobilenet.yaml"],
    # capture_output=False streams stdout/stderr live in Colab
)

print("-" * 60)
if result.returncode == 0:
    print("\n✅ Training selesai!")
    runs = sorted(model_dir.glob("mobilenet_v2_*/"))
    if runs:
        latest = runs[-1]
        print(f"📁 Model saved: {latest}")
        for f in sorted(latest.glob("**/*")):
            if f.is_file():
                kb = f.stat().st_size / 1024
                print(f"   {str(f.relative_to(model_dir)):<55} {kb:>8.1f} KB")
else:
    print(f"\n❌ Training gagal (exit code {result.returncode})")
    print("   Scroll up untuk lihat error message.")

In [ ]:
# ============================================================
# CELL 7: VERIFY MODEL OUTPUT
# Idempotent: safe untuk di-rerun kapan saja
# ============================================================
import importlib, json
from pathlib import Path

print("🔍 Verifying model artifacts...\n")

model_dir = Path("./models/final")
runs = sorted(model_dir.glob("mobilenet_v2_*/"))

if not runs:
    print("❌ No model run found — run Cell 6 first.")
else:
    latest = runs[-1]
    print(f"📁 Latest run: {latest.name}\n")

    # --- Check required artifacts ---
    required = [
        "best_model.keras",
        "training_history.json",
        "class_names.json",
    ]
    all_ok = True
    for fname in required:
        fpath = latest / fname
        if fpath.exists():
            kb = fpath.stat().st_size / 1024
            print(f"  ✅ {fname:<35} {kb:>8.1f} KB")
        else:
            print(f"  ❌ {fname:<35} MISSING")
            all_ok = False

    # --- Load and verify model ---
    model_path = latest / "best_model.keras"
    if model_path.exists():
        print("\n🧪 Loading model for shape check...")
        try:
            tf = importlib.import_module("tensorflow")
            model = tf.keras.models.load_model(str(model_path))
            print(f"  ✅ Loaded: {model_path.name}")
            print(f"  📐 Input:  {model.input_shape}")
            print(f"  📐 Output: {model.output_shape}")
            print(f"  📊 Params: {model.count_params():,}")
        except Exception as exc:
            print(f"  ❌ Load error: {exc}")

    # --- Training history summary ---
    hist_path = latest / "training_history.json"
    if hist_path.exists():
        with open(hist_path) as f:
            history = json.load(f)
        val_accs = history.get("val_accuracy", [])
        if val_accs:
            best_epoch = int(val_accs.index(max(val_accs))) + 1
            print(f"\n  🏆 Best val_accuracy: {max(val_accs):.4f}  (epoch {best_epoch}/{len(val_accs)})")
            print(f"  📈 Epoch history:")
            for e, (tr, vl) in enumerate(
                zip(history.get("accuracy", []), val_accs), start=1
            ):
                marker = " ◀ best" if e == best_epoch else ""
                print(f"     Epoch {e:>2}: train={tr:.4f}  val={vl:.4f}{marker}")

    print()
    if all_ok:
        print("✅ Model verification passed — lanjut ke Cell 8")
    else:
        print("⚠️  Some artifacts missing — check training output in Cell 6")

In [ ]:
# ============================================================
# CELL 8: EVAL — Evaluation on validation set
# Idempotent: safe untuk di-rerun
# ============================================================
import importlib, json, subprocess, sys
from pathlib import Path

print("📊 Running evaluation...\n")

model_dir = Path("./models/final")
runs = sorted(model_dir.glob("mobilenet_v2_*/"))

if not runs:
    print("❌ No model found — run Cell 6 first.")
else:
    latest = runs[-1]
    model_path = latest / "best_model.keras"

    # Try dedicated evaluate.py first
    eval_script = Path("./scripts/evaluate.py")
    if eval_script.exists():
        print(f"▶ Running {eval_script}...")
        result = subprocess.run(
            [sys.executable, str(eval_script),
             "--model", str(model_path),
             "--config", "configs/train_mobilenet.yaml"],
        )
        if result.returncode != 0:
            print(f"⚠️  evaluate.py exited with code {result.returncode}")
        else:
            print("\n✅ Evaluation done — lanjut ke Cell 9")
    else:
        # Inline evaluation
        print("📋 evaluate.py not found — running inline evaluation...\n")
        tf = importlib.import_module("tensorflow")
        np = importlib.import_module("numpy")

        model = tf.keras.models.load_model(str(model_path))

        with open(latest / "class_names.json") as f:
            class_names = json.load(f)

        print(f"  ✅ Model: {model_path.name}")
        print(f"  ✅ Classes: {class_names}")

        # Sanity check with dummy image
        dummy = np.random.rand(1, 224, 224, 3).astype("float32")
        preds = model.predict(dummy, verbose=0)
        pred_class = class_names[int(np.argmax(preds[0]))]
        confidence = float(np.max(preds[0]))
        print(f"\n  🧪 Inference sanity check:")
        print(f"     Predicted: {pred_class}  (conf: {confidence:.3f})")

        # Best val acc from history
        hist_path = latest / "training_history.json"
        if hist_path.exists():
            with open(hist_path) as f:
                history = json.load(f)
            val_accs = history.get("val_accuracy", [])
            if val_accs:
                print(f"\n  🏆 Best val_accuracy: {max(val_accs):.4f}")
                target = 0.75
                status = "✅ TARGET MET" if max(val_accs) >= target else "⚠️  Below target"
                print(f"     Target ≥ {target}: {status}")

        print("\n✅ Evaluation done — lanjut ke Cell 9")

In [ ]:
# ============================================================
# CELL 9: GRADCAM — Generate Grad-CAM visualizations
# Output: outputs/gradcam_samples/*.png
# ============================================================
import importlib, json, subprocess, sys
from pathlib import Path

print("🎨 Generating Grad-CAM visualizations...\n")
model_dir   = Path("./models/final")
runs        = sorted(model_dir.glob("mobilenet_v2_*/"))
gradcam_dir = Path("./outputs/gradcam_samples")
gradcam_dir.mkdir(parents=True, exist_ok=True)
dataset_dir = Path("./data/dataset-resized")

if not runs:
    print("❌ No model found — run Cell 6 first.")
else:
    latest     = runs[-1]
    model_path = latest / "best_model.keras"

    # --- Try generate_gradcam.py (correct script name) ---
    gradcam_script = Path("./scripts/generate_gradcam.py")
    used_script    = False

    if gradcam_script.exists() and model_path.exists():
        print(f"▶ Running {gradcam_script} — 2 images per class...")
        class_ok = 0
        for class_dir in sorted(dataset_dir.iterdir()):
            if not class_dir.is_dir():
                continue
            r = subprocess.run(
                [sys.executable, str(gradcam_script),
                 "--model-path", str(model_path),
                 "--image-dir",  str(class_dir),
                 "--output",     str(gradcam_dir),
                 "--limit",      "2"],
                capture_output=True, text=True
            )
            if r.returncode == 0:
                print(f"  ✅ {class_dir.name}: {r.stdout.strip()}")
                class_ok += 1
            else:
                print(f"  ⚠️  {class_dir.name}: {r.stderr.strip()[-300:]}")
        used_script = class_ok > 0

    # --- Inline fallback via src.gradcam (correct module path) ---
    if not used_script:
        print("📋 Inline Grad-CAM via src.gradcam.GradCAM...")
        try:
            tf          = importlib.import_module("tensorflow")
            gradcam_mod = importlib.import_module("src.gradcam")  # correct path
            GradCAM     = gradcam_mod.GradCAM

            model   = tf.keras.models.load_model(str(model_path))
            gradcam = GradCAM(model)  # auto-detect last conv layer

            # 2 images per class
            sample_images = []
            for class_dir in sorted(dataset_dir.iterdir()):
                if class_dir.is_dir():
                    sample_images.extend(sorted(class_dir.glob("*.jpg"))[:2])

            print(f"  Processing {len(sample_images)} images...")
            saved = gradcam.visualize_batch(sample_images, str(gradcam_dir))
            print(f"  ✅ Saved {len(saved)} Grad-CAM overlays")

        except Exception as exc:
            import traceback
            print(f"  ❌ Grad-CAM error: {exc}")
            traceback.print_exc()

    # --- Display preview inline ---
    pngs = sorted(gradcam_dir.glob("*.png"))[:6]
    if pngs:
        print(f"\n✅ {len(pngs)} Grad-CAM images saved to {gradcam_dir}")

        try:
            plt = importlib.import_module("matplotlib.pyplot")
            n   = min(len(pngs), 3)
            fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
            if n == 1:
                axes = [axes]
            for ax, p in zip(axes, pngs[:n]):
                ax.imshow(plt.imread(str(p)))
                ax.set_title(p.stem[:25], fontsize=8)
                ax.axis("off")
            plt.tight_layout()
            plt.savefig(str(gradcam_dir / "_preview.png"), dpi=100)
            plt.show()
        except Exception as exc:
            print(f"  ⚠️  Preview: {exc}")
    else:
        print("⚠️  No PNG generated — cek error messages di atas")



In [ ]:
# ============================================================
# CELL 10: API — Test inference engine (optional)
# Pakai src.inference langsung, tanpa start server
# ============================================================
import importlib, json, os, sys, tempfile
from pathlib import Path

print("🔌 Testing inference engine...\n")

model_dir = Path("./models/final")
runs = sorted(model_dir.glob("mobilenet_v2_*/"))

if not runs:
    print("⚠️  No model found — skip this cell.")
else:
    latest = runs[-1]
    model_path = latest / "best_model.keras"

    try:
        infer_mod = importlib.import_module("src.inference")

        # Handle different class names that might exist
        EngineClass = getattr(
            infer_mod,
            "InferenceEngine",
            getattr(infer_mod, "Classifier", None)
        )
        if EngineClass is None:
            raise AttributeError("No InferenceEngine or Classifier class in src.inference")

        engine = EngineClass(str(model_path))
        print(f"  ✅ InferenceEngine loaded: {model_path.name}")

        # Create a dummy test image
        np = importlib.import_module("numpy")
        PIL_Image = importlib.import_module("PIL.Image")

        dummy_img = PIL_Image.Image.fromarray(
            (np.random.rand(224, 224, 3) * 255).astype("uint8")
        )
        with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmp:
            dummy_img.save(tmp.name)
            tmp_path = tmp.name

        try:
            result = engine.predict(tmp_path)
            print(f"  ✅ Inference OK")
            if isinstance(result, dict):
                for k, v in result.items():
                    print(f"     {k}: {v}")
            else:
                print(f"     Result: {result}")
        finally:
            os.unlink(tmp_path)

    except ModuleNotFoundError as exc:
        print(f"  ⚠️  src.inference not found: {exc}")
        print("      Skipping — this is optional.")
    except AttributeError as exc:
        print(f"  ⚠️  API class not found: {exc}")
        print("      Skipping — model training is what matters.")
    except Exception as exc:
        print(f"  ⚠️  Inference test error: {exc}")
        print("      Model is saved and valid — API test is optional.")

print("\n✅ Lanjut ke Cell 11 untuk download")

In [ ]:
# ============================================================
# CELL 11: DOWNLOAD — Package results and download
# Creates a zip with: model, grad-cam, logs, config
# ============================================================
import json, os, shutil
from datetime import datetime
from pathlib import Path

print("📦 Packaging results for download...\n")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_name = f"waste_clf_results_{timestamp}"
staging = Path(f"/content/{zip_name}")
staging.mkdir(parents=True, exist_ok=True)

# 1. Latest model run
model_dir = Path("./models/final")
runs = sorted(model_dir.glob("mobilenet_v2_*/"))
if runs:
    latest = runs[-1]
    dest = staging / "model" / latest.name
    dest.mkdir(parents=True, exist_ok=True)
    for f in latest.glob("**/*"):
        if f.is_file():
            rel = f.relative_to(latest)
            (dest / rel).parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dest / rel)
    size_mb = sum(f.stat().st_size for f in dest.glob("**/*") if f.is_file()) / 1e6
    print(f"  ✅ Model: {latest.name}  ({size_mb:.1f} MB)")
else:
    print("  ⚠️  No model run found")

# 2. Grad-CAM images
gradcam_dir = Path("./outputs/gradcam_samples")
if gradcam_dir.exists():
    dest_gc = staging / "gradcam_samples"
    shutil.copytree(gradcam_dir, dest_gc, dirs_exist_ok=True)
    n_pngs = len(list(dest_gc.glob("*.png")))
    print(f"  ✅ Grad-CAM: {n_pngs} PNG files")
else:
    print("  ⚠️  No Grad-CAM outputs found")

# 3. Training config
cfg_src = Path("./configs/train_mobilenet.yaml")
if cfg_src.exists():
    shutil.copy2(cfg_src, staging / "train_config.yaml")
    print("  ✅ Config: train_mobilenet.yaml")

# 4. Logs
for log in Path(".").glob("*.log"):
    shutil.copy2(log, staging / log.name)
    print(f"  ✅ Log: {log.name}")

# 5. Summary JSON
summary = {"timestamp": timestamp, "runs": [r.name for r in runs]}
if runs:
    hist_path = runs[-1] / "training_history.json"
    if hist_path.exists():
        with open(hist_path) as f:
            h = json.load(f)
        val_accs = h.get("val_accuracy", [])
        if val_accs:
            summary["best_val_accuracy"] = round(max(val_accs), 4)
            summary["epochs_completed"] = len(val_accs)

with open(staging / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"  ✅ Summary: {summary}")

# Create zip
print(f"\n🗜️  Creating archive...")
zip_path = Path(f"/content/{zip_name}.zip")
shutil.make_archive(
    str(zip_path.with_suffix("")),
    "zip",
    str(staging.parent),
    zip_name
)
zip_mb = zip_path.stat().st_size / 1e6
print(f"✅ Archive: {zip_path.name}  ({zip_mb:.1f} MB)")

# Download in Colab
try:
    from google.colab import files
    print(f"\n⬇️  Triggering download of {zip_path.name}...")
    files.download(str(zip_path))
    print("✅ Download triggered!")
except ImportError:
    print(f"\n📋 Not running in Colab.")
    print(f"   File at: {zip_path}")
    print("   Use scp/sftp or copy manually.")

print("\n🎉 SELESAI! Semua hasil sudah di-package.")
best = summary.get('best_val_accuracy', 'N/A')
epochs = summary.get('epochs_completed', 'N/A')
print(f"   Best val_accuracy: {best} ({epochs} epochs)")